# Advanced Module 5 · Multi-Agent Orchestration
Connecting the agents from Module 4 — an orchestrator that routes, dispatches in parallel,
runs debate/consensus, and handles failures gracefully.

---
> **Prerequisite:** Complete Module 4 (Real-World Agents) — we reuse those agent definitions here.  
> **Setup:** Run the first two cells before anything else.

In [1]:
%pip install -q google-genai groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, time, re, getpass, random, threading, string
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from google import genai
from google.genai import types
from groq import Groq

load_dotenv()
gemini_api_key  = os.environ.get('GEMINI_API_KEY') or getpass.getpass('Paste your Gemini API key: ')
groq_api_key    = (os.environ.get('Groq_key') or getpass.getpass('Paste your Groq API key: ')).strip()

gemini_client     = genai.Client(api_key=gemini_api_key)
groq_client       = Groq(api_key=groq_api_key)
GEMINI_MODEL      = 'gemini-3.1-flash-lite'
GROQ_FAST_MODEL   = 'llama-3.1-8b-instant'
GROQ_REASON_MODEL = 'qwen/qwen3-32b'
DEFAULT_MAX_OUTPUT_TOKENS = 2048

def make_generation_config(**kwargs):
    """Build a GenerateContentConfig with sensible defaults.

    temperature=0.0  → deterministic; best for routing and tool calls.
    temperature=0.7  → creative; best for synthesis and debate arguments.
    """
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX_OUTPUT_TOKENS)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def extract_text_from_response(response) -> str:
    """Pull the final answer text from a Gemini response, skipping reasoning thoughts."""
    if response.text:
        return response.text.strip()
    text_parts = []
    for candidate in (response.candidates or []):
        if candidate.content:
            for part in candidate.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    text_parts.append(part.text)
    return ''.join(text_parts).strip()

def call_with_retry(api_function, *args, max_retries=5, **kwargs):
    """Wrap an API call with automatic retry for rate-limit and server errors."""
    for attempt in range(max_retries):
        try:
            return api_function(*args, **kwargs)
        except Exception as error:
            error_message = str(error)
            if '429' in error_message or 'RESOURCE_EXHAUSTED' in error_message:
                retry_wait_match = re.search(r'retry[^0-9]*([0-9]+)s', error_message, re.I)
                wait_seconds = int(retry_wait_match.group(1)) + 5 if retry_wait_match else 35
                print(f'  ⏳ Rate-limited — waiting {wait_seconds}s')
                time.sleep(wait_seconds)
            elif '500' in error_message or 'INTERNAL' in error_message:
                time.sleep(10 * (attempt + 1))
            else:
                raise
    raise RuntimeError('Max retries exceeded')

original_generate_content = gemini_client.models.generate_content
gemini_client.models.generate_content = lambda *args, **kwargs: call_with_retry(original_generate_content, *args, **kwargs)

print('✅ Setup complete | Gemini:', GEMINI_MODEL)

✅ Setup complete | Gemini: gemini-3.1-flash-lite


In [3]:
try:
    test_response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents='Reply with exactly the words: Hello LLM course',
        config=make_generation_config(temperature=0.0)
    )
    print('✅ API working:', extract_text_from_response(test_response))
except Exception as error:
    print('❌ Error:', error)

✅ API working: Hello LLM course


---
## The Architecture: One Orchestrator, Many Workers

Each Module 4 agent is now a **worker** — a callable black box that takes a task and returns a result.  
The **orchestrator** is a new LLM call that: classifies the user's intent, decides which worker(s) to call,  
sequences or parallelizes them, handles failures, and synthesizes the final response.

```
┌──────────────────────────────────────────────────────────────────┐
│                    MULTI-AGENT ARCHITECTURE                      │
│                                                                  │
│  User Request: "Help me plan a business trip and check if my     │
│                order ORD-1002 will arrive before I leave."       │
│                         │                                        │
│                         ▼                                        │
│              ┌──────────────────────┐                            │
│              │    ORCHESTRATOR      │  ← classifies intent,      │
│              │   (Gemini / Groq)    │    decides routing         │
│              └──────────┬───────────┘                            │
│                         │                                        │
│           ┌─────────────┼─────────────┐                          │
│           ▼             ▼             ▼                          │
│   ┌──────────────┐ ┌──────────┐ ┌──────────────┐                │
│   │   Booking    │ │ Support  │ │  Research    │                │
│   │    Agent     │ │  Agent   │ │   Agent      │                │
│   └──────┬───────┘ └────┬─────┘ └──────┬───────┘                │
│          │              │              │                         │
│          └──────────────┴──────────────┘                         │
│                         │                                        │
│                         ▼                                        │
│              ┌──────────────────────┐                            │
│              │    ORCHESTRATOR      │  ← merges results,         │
│              │  (synthesis step)    │    writes final answer     │
│              └──────────────────────┘                            │
└──────────────────────────────────────────────────────────────────┘
```

In [4]:
# ── Rebuild the 3 worker agents from Module 4 as callable functions ───────
# Each agent is defined by: its tool functions + a tool schema + a tool map.
# The agent runner (below) takes any of these three sets and runs the ReAct loop.

# ── Booking Agent tools ────────────────────────────────────────────────────
def search_flights(origin: str, destination: str, date: str) -> dict:
    return {'route': f'{origin}→{destination}', 'date': date, 'options': [
        {'flight_id': 'SK201', 'airline': 'SkyWings', 'price_usd': 580,  'duration_h': 9},
        {'flight_id': 'SW900', 'airline': 'SwiftAir', 'price_usd': 520,  'duration_h': 10},
        {'flight_id': 'LX101', 'airline': 'LuxAir',   'price_usd': 1200, 'duration_h': 8},
    ]}

def search_hotels(city: str, checkin: str, checkout: str) -> dict:
    return {'city': city, 'checkin': checkin, 'checkout': checkout, 'options': [
        {'hotel_id': 'H01', 'name': 'CityInn',        'price_per_night_usd': 85,  'rating': 3.8, 'stars': 3},
        {'hotel_id': 'H02', 'name': 'ParkView Grand', 'price_per_night_usd': 140, 'rating': 4.5, 'stars': 4},
        {'hotel_id': 'H04', 'name': 'Royal Suite',    'price_per_night_usd': 350, 'rating': 4.9, 'stars': 5},
    ]}

def book_itinerary(flight_id: str, hotel_id: str, passenger_name: str) -> dict:
    booking_ref = ''.join(random.choices(string.ascii_uppercase + string.digits, k=8))
    flight_prices = {'SK201': 580, 'SW900': 520, 'LX101': 1200}
    hotel_prices  = {'H01': 85, 'H02': 140, 'H04': 350}
    return {
        'booking_ref':  booking_ref,
        'status':       'CONFIRMED',
        'passenger':    passenger_name,
        'flight_cost':  flight_prices.get(flight_id, 0),
        'hotel_cost':   hotel_prices.get(hotel_id, 0),
    }

def get_exchange_rate(from_currency: str, to_currency: str) -> dict:
    exchange_rates = {'USD': 1.0, 'EUR': 0.93, 'GBP': 0.79, 'INR': 83.5}
    rate = exchange_rates.get(to_currency.upper(), 1.0) / exchange_rates.get(from_currency.upper(), 1.0)
    return {'from': from_currency, 'to': to_currency, 'rate': round(rate, 4)}

# ── Support Agent tools ────────────────────────────────────────────────────
def lookup_order(order_id: str) -> dict:
    order_database = {
        'ORD-1001': {'status': 'delivered',  'item': 'Laptop Pro 15"',      'price': 1299, 'delivery_date': '2025-07-10'},
        'ORD-1002': {'status': 'in_transit', 'item': 'Wireless Headphones', 'price': 249,  'eta': '2025-07-20'},
        'ORD-1003': {'status': 'cancelled',  'item': 'Smart Watch',         'price': 399},
    }
    return order_database.get(order_id, {'error': f'Order {order_id} not found'})

def check_refund_policy(reason: str) -> dict:
    refund_policies = {
        'defective':  {'eligible': True,  'method': 'full refund'},
        'not_needed': {'eligible': True,  'method': 'refund minus 10% restocking fee'},
        'late':       {'eligible': True,  'method': 'full refund'},
        'other':      {'eligible': False, 'method': 'escalate to human agent'},
    }
    return refund_policies.get(reason.lower(), refund_policies['other'])

def raise_ticket(order_id: str, issue_type: str, description: str) -> dict:
    return {'ticket_id': f'TKT-{random.randint(10000, 99999)}', 'status': 'open', 'sla_hours': 24}

def escalate_to_human(reason: str, order_id: str) -> dict:
    return {
        'escalated':      True,
        'queue':          'tier-2-support',
        'wait_time_min':  15,
        'message':        f'Escalated {order_id} for: {reason}',
    }

# ── Research Agent tools ───────────────────────────────────────────────────
ARTICLE_DATABASE = {
    'https://techblog.example.com/ai-agents-2025': {
        'title': 'AI Agents 2025',
        'text':  'AI agents handle 40% of support queries. Multi-agent systems show 3x productivity gains. Hallucination rates 8-12%.',
    },
    'https://research.example.com/llm-productivity': {
        'title': 'LLM Productivity',
        'text':  'Developers 55% faster with AI assistants. Code review time down 30%. 25% of startup codebases AI-generated.',
    },
    'https://safety.example.com/agent-risks': {
        'title': 'Agent Safety Risks',
        'text':  'Prompt injection on 15 production systems. Unrestricted agents caused $2.3M costs. Allowlists recommended.',
    },
}

def list_available_articles(topic: str) -> dict:
    return {'topic': topic, 'urls': list(ARTICLE_DATABASE.keys())}

def fetch_article(url: str) -> dict:
    article = ARTICLE_DATABASE.get(url)
    return {'url': url, 'title': article['title'], 'content': article['text']} if article else {'error': 'not found'}

def extract_key_claims(text: str) -> dict:
    """Use the fast Groq model to extract key factual claims from a block of text."""
    groq_response = groq_client.chat.completions.create(
        model=GROQ_FAST_MODEL,
        messages=[{'role': 'user', 'content': f'Extract 2-3 key factual claims as JSON array:\n{text}'}],
        max_tokens=200,
        temperature=0.0
    )
    raw_content = groq_response.choices[0].message.content
    try:
        json_match = re.search(r'\[.*?\]', raw_content, re.DOTALL)
        claims = json.loads(json_match.group(0)) if json_match else [raw_content]
    except Exception:
        claims = [raw_content]
    return {'claims': claims}

def fact_check(claim: str) -> dict:
    """Use the fast Groq model to rate a claim as LIKELY_TRUE, UNCERTAIN, or LIKELY_FALSE."""
    groq_response = groq_client.chat.completions.create(
        model=GROQ_FAST_MODEL,
        messages=[{
            'role': 'user',
            'content': (
                f'Rate as LIKELY_TRUE/UNCERTAIN/LIKELY_FALSE with a 1-sentence reason.\n'
                f'Claim: {claim}\n'
                f'JSON format: {{"verdict":"...","reason":"..."}}'
            )
        }],
        max_tokens=100,
        temperature=0.0
    )
    raw_content = groq_response.choices[0].message.content
    try:
        json_match = re.search(r'\{.*?\}', raw_content, re.DOTALL)
        result = json.loads(json_match.group(0)) if json_match else {'verdict': 'UNCERTAIN', 'reason': raw_content}
    except Exception:
        result = {'verdict': 'UNCERTAIN', 'reason': raw_content[:80]}
    return {**result, 'claim': claim}

# ── Tool schemas ───────────────────────────────────────────────────────────
def _string_schema(description: str = '') -> types.Schema:
    """Shorthand for a STRING schema field with an optional description."""
    return types.Schema(type=types.Type.STRING, description=description)

BOOKING_TOOL_SCHEMA = types.Tool(function_declarations=[
    types.FunctionDeclaration(name='search_flights', description='Search available flights.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['origin', 'destination', 'date'],
            properties={
                'origin':      _string_schema('Departure city'),
                'destination': _string_schema('Arrival city'),
                'date':        _string_schema('YYYY-MM-DD'),
            })),
    types.FunctionDeclaration(name='search_hotels', description='Search hotels in a city.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['city', 'checkin', 'checkout'],
            properties={
                'city':     _string_schema(),
                'checkin':  _string_schema('YYYY-MM-DD'),
                'checkout': _string_schema('YYYY-MM-DD'),
            })),
    types.FunctionDeclaration(name='book_itinerary', description='Confirm flight and hotel booking.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['flight_id', 'hotel_id', 'passenger_name'],
            properties={
                'flight_id':      _string_schema(),
                'hotel_id':       _string_schema(),
                'passenger_name': _string_schema(),
            })),
    types.FunctionDeclaration(name='get_exchange_rate', description='Get exchange rate between currencies.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['from_currency', 'to_currency'],
            properties={
                'from_currency': _string_schema(),
                'to_currency':   _string_schema(),
            })),
])
BOOKING_TOOL_MAP = {
    'search_flights':   search_flights,
    'search_hotels':    search_hotels,
    'book_itinerary':   book_itinerary,
    'get_exchange_rate': get_exchange_rate,
}

SUPPORT_TOOL_SCHEMA = types.Tool(function_declarations=[
    types.FunctionDeclaration(name='lookup_order', description='Look up order by ID.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['order_id'],
            properties={'order_id': _string_schema()})),
    types.FunctionDeclaration(name='check_refund_policy', description='Check refund policy for a reason.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['reason'],
            properties={'reason': _string_schema()})),
    types.FunctionDeclaration(name='raise_ticket', description='Create a support ticket.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['order_id', 'issue_type', 'description'],
            properties={
                'order_id':    _string_schema(),
                'issue_type':  _string_schema(),
                'description': _string_schema(),
            })),
    types.FunctionDeclaration(name='escalate_to_human', description='Escalate issue to a human agent.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['reason', 'order_id'],
            properties={
                'reason':   _string_schema('Why escalation is needed'),
                'order_id': _string_schema(),
            })),
])
SUPPORT_TOOL_MAP = {
    'lookup_order':        lookup_order,
    'check_refund_policy': check_refund_policy,
    'raise_ticket':        raise_ticket,
    'escalate_to_human':   escalate_to_human,
}

RESEARCH_TOOL_SCHEMA = types.Tool(function_declarations=[
    types.FunctionDeclaration(name='list_available_articles', description='List available article URLs.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['topic'],
            properties={'topic': _string_schema()})),
    types.FunctionDeclaration(name='fetch_article', description='Fetch article text by URL.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['url'],
            properties={'url': _string_schema()})),
    types.FunctionDeclaration(name='extract_key_claims', description='Extract key claims from text.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['text'],
            properties={'text': _string_schema()})),
    types.FunctionDeclaration(name='fact_check', description='Fact-check a single claim.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['claim'],
            properties={'claim': _string_schema()})),
])
RESEARCH_TOOL_MAP = {
    'list_available_articles': list_available_articles,
    'fetch_article':           fetch_article,
    'extract_key_claims':      extract_key_claims,
    'fact_check':              fact_check,
}

print('✅ All 3 worker agents rebuilt')

✅ All 3 worker agents rebuilt


In [5]:
def run_agent(
    task: str,
    system_prompt: str,
    tool_schema: types.Tool,
    tool_map: dict,
    max_steps: int = 10,
    label: str = 'Agent',
    verbose: bool = True
) -> str:
    """Generic ReAct agent runner — copied from Module 4 for self-containedness.

    All three worker agents (booking, support, research) share this loop.
    The orchestrator calls this function, passing each agent's unique
    system_prompt, tool_schema, and tool_map.
    """
    conversation = [types.Content(role='user', parts=[types.Part(
        text=f'[SYSTEM]: {system_prompt}\n\n[TASK]: {task}'
    )])]
    total_tokens = 0

    if verbose:
        print(f'\n  🤖 {label} started: {task[:80]}...' if len(task) > 80 else f'\n  🤖 {label}: {task}')

    for step_number in range(1, max_steps + 1):
        llm_response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=conversation,
            config=make_generation_config(tools=[tool_schema], temperature=0.2)
        )
        total_tokens += llm_response.usage_metadata.total_token_count

        all_function_calls = [
            part.function_call
            for part in llm_response.candidates[0].content.parts
            if hasattr(part, 'function_call') and part.function_call
        ]

        if not all_function_calls:
            final_answer = extract_text_from_response(llm_response)
            if verbose:
                print(f'  [Step {step_number}] ✅ Done | Tokens: {total_tokens}')
            return final_answer

        tool_results = []
        for function_call in all_function_calls:
            try:
                result = tool_map[function_call.name](**dict(function_call.args))
            except Exception as error:
                result = {'error': str(error)}
            tool_results.append(result)
            if verbose:
                print(f'  [Step {step_number}] 🔧 {function_call.name}({dict(function_call.args)}) → {str(result)[:80]}')

        conversation.append(llm_response.candidates[0].content)  # preserves thought_signature
        conversation.append(types.Content(role='user', parts=[
            types.Part(function_response=types.FunctionResponse(
                name=function_call.name,
                response={'result': result}
            ))
            for function_call, result in zip(all_function_calls, tool_results)
        ]))

    return 'Max steps reached.'

print('✅ Agent runner ready')

✅ Agent runner ready


---
## Demo 1: Orchestrator Routing

An ambiguous user message — the orchestrator classifies intent and decides which agent(s) to invoke.

In [6]:
# The agent registry is the single source of truth for all available worker agents.
# Each entry maps an agent name → its description, tool schema, tool map, and system prompt.
# The orchestrator reads descriptions to decide which agents to invoke, then uses
# schema/map/system to actually run them.

AGENT_REGISTRY = {
    'booking_agent': {
        'description': 'Handles flight search, hotel search, booking, and trip cost calculations.',
        'schema':      BOOKING_TOOL_SCHEMA,
        'map':         BOOKING_TOOL_MAP,
        'system':      'You are a smart travel booking assistant. Search flights and hotels, then book the best option.',
    },
    'support_agent': {
        'description': 'Handles customer support: order lookup, refunds, tickets, and escalations.',
        'schema':      SUPPORT_TOOL_SCHEMA,
        'map':         SUPPORT_TOOL_MAP,
        'system':      'You are a customer support agent. Be empathetic. Look up orders. Escalate complex cases.',
    },
    'research_agent': {
        'description': 'Fetches articles, extracts claims, fact-checks them, and writes research reports.',
        'schema':      RESEARCH_TOOL_SCHEMA,
        'map':         RESEARCH_TOOL_MAP,
        'system':      'You are a research analyst. Fetch articles, extract claims, fact-check, write structured reports.',
    },
}

def orchestrate(user_request: str, verbose: bool = True) -> str:
    """Route a user request to the appropriate worker agent(s) and synthesize their outputs.

    Step 1 — Routing: send the user request + agent descriptions to the LLM.
             The LLM returns a JSON routing decision: which agents to invoke and
             what subtask to give each one.

    Step 2 — Dispatch: run each selected agent sequentially with its subtask.

    Step 3 — Synthesis: send all agent outputs back to the LLM and ask it to
             merge them into a single coherent final response.
    """
    agent_descriptions = '\n'.join(
        f'- {agent_name}: {agent_meta["description"]}'
        for agent_name, agent_meta in AGENT_REGISTRY.items()
    )

    routing_prompt = f"""You are an orchestrator. Given a user request, output a JSON object:
{{"agents": [list of agent names to invoke], "subtasks": {{agent_name: subtask_for_that_agent}}}}

Available agents:
{agent_descriptions}

User request: {user_request}

Rules:
- Only include agents that are actually needed
- The subtask should be a concise task description for that agent
- Return ONLY the JSON, no explanation"""

    routing_response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=routing_prompt,
        config=make_generation_config(temperature=0.0)
    )
    routing_text = extract_text_from_response(routing_response)

    # Parse the routing decision JSON
    try:
        json_match      = re.search(r'\{.*\}', routing_text, re.DOTALL)
        routing_decision = json.loads(json_match.group(0)) if json_match else {'agents': [], 'subtasks': {}}
    except Exception:
        routing_decision = {'agents': [], 'subtasks': {}}

    if verbose:
        print('═' * 65)
        print(f'USER: {user_request}')
        print('═' * 65)
        print(f'\n🎯 ORCHESTRATOR ROUTING DECISION:')
        print(f'   Agents selected: {routing_decision.get("agents", [])}')
        for agent_name, subtask in routing_decision.get('subtasks', {}).items():
            print(f'   {agent_name} → "{subtask}"')

    # Dispatch each selected agent and collect their results
    agent_outputs = {}
    for agent_name in routing_decision.get('agents', []):
        if agent_name not in AGENT_REGISTRY:
            continue
        agent_meta  = AGENT_REGISTRY[agent_name]
        agent_task  = routing_decision.get('subtasks', {}).get(agent_name, user_request)
        agent_result = run_agent(
            task=agent_task,
            system_prompt=agent_meta['system'],
            tool_schema=agent_meta['schema'],
            tool_map=agent_meta['map'],
            label=agent_name,
            verbose=verbose
        )
        agent_outputs[agent_name] = agent_result

    if not agent_outputs:
        return 'No agents were invoked. Please rephrase your request.'

    # Synthesis: merge all agent outputs into one final response
    synthesis_prompt = (
        f'Original user request: {user_request}\n\n'
        + '\n\n'.join(
            f'Output from {agent_name}:\n{agent_result}'
            for agent_name, agent_result in agent_outputs.items()
        )
        + '\n\nSynthesize a clear, complete final response to the user.'
    )
    synthesis_response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=synthesis_prompt,
        config=make_generation_config(temperature=0.3)
    )
    final_answer = extract_text_from_response(synthesis_response)

    if verbose:
        print('\n' + '═' * 65)
        print('🏁 ORCHESTRATOR FINAL RESPONSE:')
        print('═' * 65)
        print(final_answer)

    return final_answer

orchestrate(
    "Help me plan a business trip from Mumbai to Berlin next month. "
    "Also, check what's happening with my order ORD-1002."
)

═════════════════════════════════════════════════════════════════
USER: Help me plan a business trip from Mumbai to Berlin next month. Also, check what's happening with my order ORD-1002.
═════════════════════════════════════════════════════════════════

🎯 ORCHESTRATOR ROUTING DECISION:
   Agents selected: ['booking_agent', 'support_agent']
   booking_agent → "Plan a business trip from Mumbai to Berlin for next month, including flight and hotel options."
   support_agent → "Look up the status of order ORD-1002."

  🤖 booking_agent started: Plan a business trip from Mumbai to Berlin for next month, including flight and ...
  [Step 1] 🔧 search_flights({'date': '2025-06-15', 'origin': 'Mumbai', 'destination': 'Berlin'}) → {'route': 'Mumbai→Berlin', 'date': '2025-06-15', 'options': [{'flight_id': 'SK20
  [Step 2] 🔧 search_hotels({'checkin': '2025-06-16', 'city': 'Berlin', 'checkout': '2025-06-20'}) → {'city': 'Berlin', 'checkin': '2025-06-16', 'checkout': '2025-06-20', 'options':
  [Step 3

'Here is the complete update regarding your business trip and order status:\n\n### **Business Trip Planning: Mumbai to Berlin (June 15th – June 20th)**\nI have researched flight and hotel options for your upcoming trip. Based on a balance of cost, travel time, and business suitability, I recommend the following:\n\n*   **Flight:** **SkyWings (SK201)** – $580 (9-hour duration).\n*   **Hotel:** **ParkView Grand** – $140/night (4.5 rating, excellent for business travelers).\n\n**Other options:**\n*   **Flights:** LuxAir ($1,200, 8 hours) or SwiftAir ($520, 10 hours).\n*   **Hotels:** Royal Suite ($350/night, luxury) or CityInn ($85/night, budget).\n\n**Next Steps:** If you would like me to proceed with these bookings, please provide the passenger name, and I will finalize the arrangements for you.\n\n***\n\n### **Order Status: ORD-1002**\nRegarding your order for the Wireless Headphones (ORD-1002), I have confirmed that your package is currently in transit. It is scheduled to arrive on **

---
## Demo 2: Parallel Worker Dispatch

When two agents can run independently, dispatch them in parallel using threads.  
Print the wall-clock time saved vs. running them sequentially.

In [7]:
def run_agents_in_parallel(agent_tasks: dict, verbose: bool = True) -> dict:
    """Run multiple agents concurrently using a thread pool.

    agent_tasks: {agent_name: task_string, ...}
    Returns:     {agent_name: result_string, ...}

    Each agent runs in its own thread. Wall-clock time equals the slowest single
    agent rather than the sum of all agents — which is the key scalability gain
    in multi-agent systems.
    """
    results = {}

    def run_one_agent(agent_name: str, task: str):
        agent_meta = AGENT_REGISTRY[agent_name]
        return agent_name, run_agent(
            task=task,
            system_prompt=agent_meta['system'],
            tool_schema=agent_meta['schema'],
            tool_map=agent_meta['map'],
            label=agent_name,
            verbose=verbose
        )

    with ThreadPoolExecutor(max_workers=len(agent_tasks)) as thread_pool:
        futures = {
            thread_pool.submit(run_one_agent, agent_name, task): agent_name
            for agent_name, task in agent_tasks.items()
        }
        for completed_future in as_completed(futures):
            agent_name, agent_result = completed_future.result()
            results[agent_name] = agent_result

    return results

# ── Timing comparison: sequential vs. parallel ─────────────────────────────
booking_task  = 'I want to fly from Delhi to Tokyo on 2025-09-01 for 2 nights. Find cheapest options.'
research_task = 'Research the current state of AI agents in 2025. Summarize key findings.'

print('Running agents SEQUENTIALLY...')
sequential_start_time = time.time()
sequential_booking_result  = run_agent(booking_task,  AGENT_REGISTRY['booking_agent']['system'],
                                       BOOKING_TOOL_SCHEMA, BOOKING_TOOL_MAP, label='booking_agent', verbose=False)
sequential_research_result = run_agent(research_task, AGENT_REGISTRY['research_agent']['system'],
                                       RESEARCH_TOOL_SCHEMA, RESEARCH_TOOL_MAP, label='research_agent',
                                       verbose=False, max_steps=8)
sequential_elapsed_seconds = round(time.time() - sequential_start_time, 2)
print(f'Sequential time: {sequential_elapsed_seconds}s')

print('\nRunning agents IN PARALLEL...')
parallel_start_time = time.time()
parallel_results = run_agents_in_parallel({
    'booking_agent':  booking_task,
    'research_agent': research_task,
}, verbose=True)
parallel_elapsed_seconds = round(time.time() - parallel_start_time, 2)

print(f'\nParallel time: {parallel_elapsed_seconds}s')
print('\n' + '=' * 65)
print('TIMING COMPARISON')
print(f'  Sequential : {sequential_elapsed_seconds}s')
print(f'  Parallel   : {parallel_elapsed_seconds}s')
speedup_factor = round(sequential_elapsed_seconds / max(parallel_elapsed_seconds, 0.1), 2)
time_saved_seconds = round(sequential_elapsed_seconds - parallel_elapsed_seconds, 2)
print(f'  Speedup    : {speedup_factor}x faster')
print(f'  Saved      : {time_saved_seconds}s')
print('\nParallel dispatch is the key to scalable multi-agent systems.')

Running agents SEQUENTIALLY...
  ⏳ Rate-limited — waiting 51s
Sequential time: 61.29s

Running agents IN PARALLEL...

  🤖 booking_agent started: I want to fly from Delhi to Tokyo on 2025-09-01 for 2 nights. Find cheapest opti...

  🤖 research_agent: Research the current state of AI agents in 2025. Summarize key findings.
  [Step 1] 🔧 list_available_articles({'topic': 'AI agents 2025 state of the art trends'}) → {'topic': 'AI agents 2025 state of the art trends', 'urls': ['https://techblog.e
  [Step 1] 🔧 search_flights({'origin': 'Delhi', 'date': '2025-09-01', 'destination': 'Tokyo'}) → {'route': 'Delhi→Tokyo', 'date': '2025-09-01', 'options': [{'flight_id': 'SK201'
  [Step 2] 🔧 fetch_article({'url': 'https://techblog.example.com/ai-agents-2025'}) → {'url': 'https://techblog.example.com/ai-agents-2025', 'title': 'AI Agents 2025'
  [Step 2] 🔧 search_hotels({'checkout': '2025-09-03', 'city': 'Tokyo', 'checkin': '2025-09-01'}) → {'city': 'Tokyo', 'checkin': '2025-09-01', 'checkout': '2025-

---
## Demo 3: Debate + Consensus (with Reasoning Model as Judge)

Two instances of the Research Agent argue opposite sides of a decision.  
A Judge agent (Qwen3 reasoning model) weighs both arguments and picks the stronger one —  
showing the thinking trace.

This pattern is used in: agentic code review, investment analysis, risk assessment.

In [8]:
DEBATE_TOPIC = "For a 500-person engineering org, should we adopt microservices or stick with a monolith?"

PRO_MICROSERVICES_SYSTEM_PROMPT = (
    'You are arguing STRONGLY IN FAVOR of microservices architecture. '
    'Make the best possible case with concrete benefits, scalability advantages, and industry examples. '
    'Do NOT mention any downsides.'
)
PRO_MONOLITH_SYSTEM_PROMPT = (
    'You are arguing STRONGLY IN FAVOR of a monolith architecture. '
    'Make the best possible case: simplicity, operational ease, faster development, and real-world successes. '
    'Do NOT mention any downsides.'
)

print('═' * 65)
print('DEBATE TOPIC:', DEBATE_TOPIC)
print('═' * 65)

# Run both debaters in parallel using threads so neither waits for the other
microservices_argument = None
monolith_argument      = None

def fetch_microservices_argument():
    global microservices_argument
    debate_response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=f'[SYSTEM]: {PRO_MICROSERVICES_SYSTEM_PROMPT}\n\n[TOPIC]: {DEBATE_TOPIC}',
        config=make_generation_config(temperature=0.7, max_output_tokens=512)
    )
    microservices_argument = extract_text_from_response(debate_response)

def fetch_monolith_argument():
    global monolith_argument
    debate_response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=f'[SYSTEM]: {PRO_MONOLITH_SYSTEM_PROMPT}\n\n[TOPIC]: {DEBATE_TOPIC}',
        config=make_generation_config(temperature=0.7, max_output_tokens=512)
    )
    monolith_argument = extract_text_from_response(debate_response)

microservices_thread = threading.Thread(target=fetch_microservices_argument)
monolith_thread      = threading.Thread(target=fetch_monolith_argument)
microservices_thread.start()
monolith_thread.start()
microservices_thread.join()
monolith_thread.join()

print('\n📌 ARGUMENT FOR MICROSERVICES:')
print(microservices_argument[:500], '...' if len(microservices_argument) > 500 else '')
print('\n📌 ARGUMENT FOR MONOLITH:')
print(monolith_argument[:500], '...' if len(monolith_argument) > 500 else '')

# Qwen3 reasoning model acts as judge — it deliberates on both arguments
# and exposes its thinking trace before delivering a verdict
print('\n' + '─' * 65)
print('🧠 JUDGE (Qwen3-32B reasoning model) deliberating...')
print('─' * 65)

judge_prompt = f"""You are a neutral technical judge evaluating two arguments about architecture choice.

TOPIC: {DEBATE_TOPIC}

ARGUMENT A (Microservices):
{microservices_argument}

ARGUMENT B (Monolith):
{monolith_argument}

Evaluate both arguments. Which is stronger given the context (500-person eng org)?
Provide:
1. Your verdict (A or B) with detailed reasoning
2. The key strengths and weaknesses of each argument
3. A nuanced recommendation for the org"""

judge_response = groq_client.chat.completions.create(
    model=GROQ_REASON_MODEL,
    messages=[{'role': 'user', 'content': judge_prompt}],
    max_tokens=2048,
    temperature=0.6
)

judge_full_content   = judge_response.choices[0].message.content
thinking_trace_match = re.search(r'<think>(.*?)</think>', judge_full_content, re.DOTALL)
thinking_trace       = thinking_trace_match.group(1).strip() if thinking_trace_match else ''
judge_verdict        = re.sub(r'<think>.*?</think>', '', judge_full_content, flags=re.DOTALL).strip()

if thinking_trace:
    print('\n[JUDGE THINKING TRACE (first 400 chars)]:')
    print(thinking_trace[:400], '...')

print('\n[JUDGE VERDICT]:')
print(judge_verdict)

═════════════════════════════════════════════════════════════════
DEBATE TOPIC: For a 500-person engineering org, should we adopt microservices or stick with a monolith?
═════════════════════════════════════════════════════════════════
  ⏳ Rate-limited — waiting 47s

📌 ARGUMENT FOR MICROSERVICES:
For an engineering organization of 500 people, the transition to a microservices architecture is not just an option—it is an **imperative for survival and market dominance.** At this scale, a monolith becomes a structural anchor, dragging down velocity, stifling innovation, and creating a single point of failure that threatens the entire business.

Here is the definitive case for why your organization must adopt a microservices architecture.

### 1. Unbounded Organizational Scalability
With 500  ...

📌 ARGUMENT FOR MONOLITH:
For a 500-person engineering organization, the decision is clear: **stick with the monolith.** The siren song of microservices is a recipe for operational chaos, whereas t

---
## Demo 4: Failure & Fallback — Resilient Orchestration

The Booking Agent hits a simulated tool failure mid-run.  
The orchestrator detects it and re-routes to the Support Agent as a fallback.

In [9]:
def search_flights_always_failing(origin: str, destination: str, date: str) -> dict:
    """Simulates a complete flight-search service outage.

    Every call raises ConnectionError. The orchestrator catches this,
    detects the failure, and re-routes to the support agent as a fallback.
    """
    raise ConnectionError(
        'OUTAGE: Flight search service is down. Please try again later or contact support.'
    )

# Override the booking tool map to use the failing version
FAILING_BOOKING_TOOL_MAP = {**BOOKING_TOOL_MAP, 'search_flights': search_flights_always_failing}

def orchestrate_with_fallback(user_request: str):
    """Orchestrate with automatic fallback: if the primary agent fails, re-route.

    This demonstrates resilient orchestration:
    1. Try the booking agent (primary).
    2. If it fails or reports an error, catch it and hand off to the support agent (fallback).
    3. The support agent apologises professionally, creates a ticket, and sets expectations.

    The user never sees a crash — they get a graceful, helpful response either way.
    """
    print('═' * 65)
    print(f'USER: {user_request}')
    print('═' * 65)

    # Attempt the primary agent
    print('\n🎯 Routing to: booking_agent (primary)')
    try:
        primary_result = run_agent(
            task=user_request,
            system_prompt=AGENT_REGISTRY['booking_agent']['system'],
            tool_schema=BOOKING_TOOL_SCHEMA,
            tool_map=FAILING_BOOKING_TOOL_MAP,   # ← uses the always-failing search_flights
            label='booking_agent (primary)',
            verbose=True,
            max_steps=3  # fail fast so we don't burn tokens on a broken agent
        )

        # Check whether the agent itself signalled failure in its text output
        if 'Max steps' in primary_result or 'OUTAGE' in primary_result or 'error' in primary_result.lower():
            raise RuntimeError(f'Booking agent reported failure: {primary_result[:100]}')

        return primary_result

    except Exception as primary_error:
        print(f'\n⚠️  PRIMARY AGENT FAILED: {primary_error}')
        print('🔄 ORCHESTRATOR: Falling back to support_agent...')

        # Build a fallback task that explains the situation to the support agent
        fallback_task = (
            f'The user tried to make a booking but our flight search system is down. '
            f'Original request: "{user_request}". '
            f'Please apologise professionally, raise a support ticket, '
            f'and let the user know we will follow up within 24 hours.'
        )
        fallback_result = run_agent(
            task=fallback_task,
            system_prompt=AGENT_REGISTRY['support_agent']['system'],
            tool_schema=SUPPORT_TOOL_SCHEMA,
            tool_map={
                **SUPPORT_TOOL_MAP,
                # Override lookup_order for this fallback — no real order to look up
                'lookup_order': lambda order_id: {'status': 'N/A', 'note': 'system outage'},
            },
            label='support_agent (fallback)',
            verbose=True
        )

        print('\n' + '═' * 65)
        print('🏁 FALLBACK RESPONSE TO USER:')
        print('═' * 65)
        print(fallback_result)
        return fallback_result

orchestrate_with_fallback(
    "I need to book a flight from Chennai to Singapore for next week, 4 nights."
)

═════════════════════════════════════════════════════════════════
USER: I need to book a flight from Chennai to Singapore for next week, 4 nights.
═════════════════════════════════════════════════════════════════

🎯 Routing to: booking_agent (primary)

  🤖 booking_agent (primary): I need to book a flight from Chennai to Singapore for next week, 4 nights.
  [Step 1] 🔧 search_flights({'destination': 'Singapore', 'origin': 'Chennai', 'date': '2025-05-19'}) → {'error': 'OUTAGE: Flight search service is down. Please try again later or cont
  [Step 2] 🔧 search_hotels({'city': 'Singapore', 'checkin': '2025-05-19', 'checkout': '2025-05-23'}) → {'city': 'Singapore', 'checkin': '2025-05-19', 'checkout': '2025-05-23', 'option
  [Step 3] ✅ Done | Tokens: 1707


'It appears that the flight search service is currently experiencing an outage, so I am unable to retrieve flight options for your trip from Chennai to Singapore at this moment.\n\nHowever, I have successfully retrieved hotel options for your 4-night stay in Singapore (May 19th – May 23rd):\n\n*   **CityInn**: $85/night (3 stars, 3.8 rating)\n*   **ParkView Grand**: $140/night (4 stars, 4.5 rating)\n*   **Royal Suite**: $350/night (5 stars, 4.9 rating)\n\nWould you like me to keep monitoring the flight service and notify you once it is back online, or would you like to proceed with a hotel booking in the meantime?'

In [10]:
# ✏️  Add a 4th custom agent to the AGENT_REGISTRY and plug it into the orchestrator.
#
#    Ideas:
#    - WeatherAgent: check weather forecast for a city using a stub tool
#    - CurrencyAgent: compare exchange rates across multiple currencies
#    - VisaAgent: check visa requirements between two countries
#
#    Steps:
#    1. Define tool function(s)
#    2. Build tool schema using gt.FunctionDeclaration
#    3. Add to AGENT_REGISTRY with a name, description, schema, map, and system prompt
#    4. Re-run orchestrate() with a task that would trigger your new agent

# Example structure:
# AGENT_REGISTRY['weather_agent'] = {
#     'description': 'Checks weather forecast for any city.',
#     'schema': my_weather_schema,
#     'map':    {'get_forecast': my_get_forecast},
#     'system': 'You are a weather information agent. Use the forecast tool to answer weather questions.'
# }

print('Define your agent, add it to AGENT_REGISTRY, then run:')
print('orchestrate("Ask something that requires your new agent")')

Define your agent, add it to AGENT_REGISTRY, then run:
orchestrate("Ask something that requires your new agent")


---
## Full-Day Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────────┐
│                  EVERYTHING YOU BUILT TODAY                         │
│                                                                     │
│  MODULE 1: Function Calling                                         │
│  ┌──────┐  tool JSON   ┌─────────┐  result   ┌───────┐             │
│  │ LLM  │────────────▶│ Python  │──────────▶│  LLM  │             │
│  └──────┘              └─────────┘           └───────┘             │
│                                                                     │
│  MODULE 2: ReAct Agent Loop                                         │
│  [Think] → [Act] → [Observe] → [Think] → ... → [Answer]            │
│                                                                     │
│  MODULE 3: MCP + Reasoning + Safety                                 │
│  MCP: standard protocol for tool discovery and invocation           │
│  Reasoning: <think>...</think> for hard problems                    │
│  Safety: allowlists stop prompt injection attacks                   │
│                                                                     │
│  MODULE 4: Specialized Workers                                      │
│  BookingAgent  ──┐                                                  │
│  SupportAgent  ──┤  (each = system prompt + tools + ReAct loop)     │
│  ResearchAgent ──┘                                                  │
│                                                                     │
│  MODULE 5: Orchestration                                            │
│                      ┌──────────────┐                               │
│                      │ ORCHESTRATOR │  classify → route → merge     │
│                      └──────┬───────┘                               │
│          ┌───────────────────┼──────────────────┐                   │
│          ▼                   ▼                  ▼                   │
│   BookingAgent        SupportAgent        ResearchAgent             │
│   (parallel ←──────────────────────────────────→ parallel)         │
│          │                   │                  │                   │
│          └───────────────────┴──────────────────┘                   │
│                              │                                      │
│                      ┌───────▼───────┐                              │
│                      │  SYNTHESIS    │  merge all outputs           │
│                      └───────────────┘                              │
│                                                                     │
│  Patterns used: routing, parallel dispatch, debate+consensus,       │
│                 failure+fallback, reasoning model as judge          │
└─────────────────────────────────────────────────────────────────────┘
```

---
## Key Takeaways

| Concept | What it means |
|---|---|
| Orchestrator | An LLM that classifies user intent and decides which worker agents to run |
| Worker agents | Specialized agents from Module 4 — callable black boxes |
| Routing | Orchestrator outputs a JSON `{agents: [...], subtasks: {...}}` decision |
| Parallel dispatch | Run independent agents in threads — drastically reduces wall-clock time |
| Debate + consensus | Run adversarial agents on opposite sides, use reasoning model as judge |
| Failure + fallback | Detect agent failures, re-route to a fallback agent — resilient systems |
| Synthesis | Final orchestrator call merges all worker outputs into a coherent response |

---
## Full-Day Summary Cheat Sheet

| Module | Key idea | When to use |
|---|---|---|
| 1 — Function Calling | LLM returns JSON; your code runs the function | Any time the model needs to access external data or systems |
| 2 — ReAct Loop | Reason → Act → Observe until done | Open-ended tasks where the number of steps is unknown in advance |
| 3 — MCP | Standard protocol; any client works with any server | Building tools that need to be reusable across models and frameworks |
| 3 — Reasoning | `<think>` trace; slower but better on hard problems | Math, logic puzzles, ambiguous instructions, complex decisions |
| 3 — Safety | Allowlist every tool call before executing | Any agent that acts on real systems — mandatory, not optional |
| 4 — Real Agents | System prompt + tools + ReAct = a complete agent | Any domain-specific automation task |
| 5 — Orchestration | One orchestrator + N workers = scalable multi-agent system | Complex tasks that need multiple specializations or parallelism |